In [1]:
import numpy as np
import pandas as pd

In [28]:
# holoc caption

In [29]:
total_df = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/haloc_extension/caption/gemini_labeled_28k.csv")
total_df = total_df[total_df["hallucination_candidates"].apply(lambda x: len(eval(x)) != 0)]

In [30]:
total_df["image_id"].nunique()

9880

In [32]:

all_data = []
for image_id, df in total_df.groupby("image_id"):
    df = df.drop_duplicates(subset=["question", "answer"])
    if df.shape[0] >= 1 and df.shape[0] < 2:
        c_data = {"image_id": df["image_id"].iloc[0], "image_path": df["image_path"].iloc[0], "qa_pairs": df[["question", "answer"]].to_dict(orient="records")}
        all_data.append(c_data)
    elif df.shape[0] >= 2:
        df = df.sample(2)
        c_data = {"image_id": df["image_id"].iloc[0], "image_path": df["image_path"].iloc[0], "qa_pairs": df[["question", "answer"]].to_dict(orient="records")}
        all_data.append(c_data)


In [33]:
final_df = pd.DataFrame(all_data)

In [34]:
final_df.to_csv("/Data2/Arun-UAV/NLP/vision_halu/hal_detection_head_train_datasets/holoc/caption/unlabeled_caption_data.csv", index=False)

In [38]:
final_df["image_id"].nunique()

9880

# instruct

In [41]:
inst_data = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/haloc_extension/instruct/gemini_labeled_40k.csv")

In [42]:
inst_data.head(5)

,image_id,question,answer,candidates,hallucination_candidates,candidates_inx,hallucination_candidates_inx,image_path,question_id
0,2317351.jpg,Describe the color of the hat.,The hat is blue.,"['blue', 'hat']",['blue'],"{'blue': [(11, 15)], 'hat': [(4, 7)]}","{'blue': [(11, 15)]}",/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,e7f137be-9190-42f0-a1b0-7486e8028fad
1,2317351.jpg,Describe the color of the hat.,The hat is gray.,"['hat', 'gray']",['gray'],"{'hat': [(4, 7)], 'gray': [(11, 15)]}","{'gray': [(11, 15)]}",/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,c9903799-0507-4102-84fe-3dcd587ae9e0
2,2376102.jpg,Identify the item located at the bottom of the...,The surfboard is in the top of the image.,"['surfboard', 'top', 'image']",['top'],"{'surfboard': [(4, 13)], 'top': [(24, 27)], 'i...","{'top': [(24, 27)]}",/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,2b6092e2-7bc6-43a2-bffe-f7963ada7a22
3,2376102.jpg,Identify the item located at the bottom of the...,The surfboard is at the bottom of the image.,"['bottom', 'surfboard', 'image']",[],"{'bottom': [(24, 30)], 'surfboard': [(4, 13)],...",{},/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,72315862-cce0-43e0-96e5-4202a1ca7fa2
4,2317132.jpg,Are there umbrellas scattered on the sand?,"No, there is a boat on the sand.","['sand', 'boat', 'no']",[],"{'sand': [(27, 31)], 'boat': [(15, 19)], 'no':...",{},/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,fad201cd-b973-4bb4-8d0c-5a33e83415b4


In [43]:
all_data = []
for image_id, df in inst_data.groupby("image_id"):
    df = df.drop_duplicates(subset=["question", "answer"])
    if df.shape[0] >= 1 and df.shape[0] < 2:
        c_data = {"image_id": df["image_id"].iloc[0], "image_path": df["image_path"].iloc[0], "qa_pairs": df[["question", "answer"]].to_dict(orient="records")}
        all_data.append(c_data)
    elif df.shape[0] >= 2:
        df = df.sample(2)
        c_data = {"image_id": df["image_id"].iloc[0], "image_path": df["image_path"].iloc[0], "qa_pairs": df[["question", "answer"]].to_dict(orient="records")}
        all_data.append(c_data)

In [46]:
inst_data["image_id"].nunique()

11312

In [50]:
total_data = pd.DataFrame(all_data)

In [53]:
total_data.to_csv("/Data2/Arun-UAV/NLP/vision_halu/hal_detection_head_train_datasets/holoc/instruct/unlabeled_instruct_data.csv", index=False)

In [54]:
# vqa

In [58]:
tn_data = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/haloc_extension/vqa/tn_data.csv")
tp_data = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/haloc_extension/vqa/tp_data.csv")

vqa_data = pd.concat([tn_data, tp_data])

In [68]:
all_data = []
for image_id in tn_data["image_id"].unique():
    
    tn_df = tn_data[tn_data["image_id"]==image_id]
    tn_df = tn_df.drop_duplicates(subset=["question", "answer"])
    
    tp_df = tp_data[tp_data["image_id"]==image_id]
    tp_df = tp_df.drop_duplicates(subset=["question", "answer"])

    if tn_df.shape[0] >= 1 and tn_df.shape[0] < 2:
        s_tn_df = tn_df
    elif tn_df.shape[0] >= 2:
        s_tn_df = tn_df.sample(2)

    if tp_df.shape[0] >= 1 and tp_df.shape[0] < 2:
        s_tp_df = tp_df
    elif tp_df.shape[0] >= 2:
        s_tp_df = tp_df.sample(2)

    if s_tn_df is not None and s_tp_df is not None:
        combined = pd.concat([s_tn_df, s_tp_df])
        combined["question"] = combined["question"].str.replace("Question: ", "")
        c_data = {"image_id": combined["image_id"].iloc[0], "image_path": combined["image_path"].iloc[0], "qa_pairs": combined[["question", "answer"]].to_dict(orient="records")}
        all_data.append(c_data)

In [69]:
len(all_data)

14804

In [70]:
total_df = pd.DataFrame(all_data)

In [75]:
total_df.to_csv("/Data2/Arun-UAV/NLP/vision_halu/hal_detection_head_train_datasets/holoc/vqa/unlabeled_vqa_data.csv", index=False)

In [76]:
# visual genome

In [21]:
vg_qa = pd.read_json("/Data2/Arun-UAV/NLP/vision_halu/visual_genome/annotations/question_answers.json")

In [22]:
import os
imgs = os.listdir("/Data2/Arun-UAV/NLP/vision_halu/visual_genome/target_images")
filt_ids = []
for img_path in imgs:
    filt_ids.append(int(img_path.split(".")[0]))

In [23]:
len(filt_ids)

26222

In [102]:
isinstance(vg_qa["id"], pd.Series)

True

In [24]:
vg_qa["id"] = vg_qa["id"].apply(lambda x: int(x.item()) if isinstance(x, pd.Series) else int(x))

In [25]:
filt_vqa_qa = vg_qa[vg_qa["id"].isin(filt_ids)]

In [26]:
filt_vqa_qa = filt_vqa_qa[filt_vqa_qa["qas"].apply(lambda x: len(x) != 0)]

In [32]:
filt_vqa_qa.head(2)

,id,qas
5006,2415080,"[{'a_objects': [], 'question': 'What is on the..."
5018,2415095,"[{'a_objects': [], 'question': 'What is on the..."


In [33]:
sample_10k_df = filt_vqa_qa.sample(10000)

In [47]:
from uuid import uuid4

In [48]:
all_res = []
for inx, row in sample_10k_df.iterrows():
    qa_df = pd.DataFrame(row["qas"]).sample(1)
    _id = row["id"]
    
    question = qa_df["question"].iloc[0]
    answer = qa_df["answer"].iloc[0]
    image_id = _id
    image_path = "/Data2/Arun-UAV/NLP/vision_halu/visual_genome/VG_100K/" + str(image_id) + ".jpg"
    all_res.append({"image_id": image_id, "image_path": image_path, "question": question, "answer": answer, "question_id": str(uuid4())})

In [49]:
final_vg_df = pd.DataFrame(all_res)

In [52]:
final_vg_df.tail(2)

,image_id,image_path,question,answer,question_id
9998,2386645,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Where is the toilet paper?,To the right of the toilet.,1d3863fc-50ee-40ab-a731-7d46b1605d6a
9999,2372546,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,What is the giraffe covered with?,Brown spots.,4ff67e96-21b1-44c0-8d90-913c29315118


In [56]:
final_vg_df.to_csv("/Data2/Arun-UAV/NLP/vision_halu/hal_detection_head_train_datasets/vga/vga_unlabeled_data_10k.csv", index=False)

In [2]:
# GQA dataset

In [3]:
gqa = pd.read_json("/Data2/Arun-UAV/NLP/vision_halu/large_image_datasets/GQA/train_balanced_questions.json",lines = True)

In [4]:
gqa.head(2)

,2930152,7333408,7333405,15736264,111007521,51004431,7452748,51004435,19274091,974084,...,15108141,5145758,13241402,13241405,8331278,18657040,8976986,18429706,13190246,5145757
0,"{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...",...,"{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende...","{'semantic': [{'operation': 'select', 'depende..."


In [6]:
gqa_columns = gqa.columns.tolist()
import random

sampled_gqa_columns = random.sample(gqa_columns, 50000)

In [7]:
len(sampled_gqa_columns)

50000

In [ ]:
from tqdm import tqdm

all_results = []
for i in tqdm(sampled_gqa_columns):
    try:
        res =  gqa[i].iloc[0]
        question = res["question"]
        imageId = res["imageId"]
        answer = res["answer"]
        fullAnswer = res["fullAnswer"]

        all_results.append({
            "question": question,
            "imageId": imageId,
            "answer": answer,
            "fullAnswer": fullAnswer
        })
    except:
        continue

  0%|          | 334/100000 [00:23<1:56:17, 14.28it/s]

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def process_item(i):
    try:
        res = gqa[i].iloc[0]
        return {
            "question": res["question"],
            "imageId": res["imageId"],
            "answer": res["answer"],
            "fullAnswer": res["fullAnswer"]
        }
    except Exception:
        return None

all_results = []
with ThreadPoolExecutor(max_workers=32) as executor:  # adjust workers based on cores
    futures = {executor.submit(process_item, i): i for i in sampled_gqa_columns}
    for future in tqdm(as_completed(futures), total=len(sampled_gqa_columns)):
        result = future.result()
        if result is not None:
            all_results.append(result)


100%|██████████| 50000/50000 [09:48<00:00, 85.00it/s] 


In [10]:
filtered_df = pd.DataFrame(all_results)

In [13]:
yn_data = filtered_df[filtered_df["answer"].apply(lambda x: "yes" in x.lower() or "no" in x.lower())]

In [67]:
data = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/hal_detection_head_train_datasets/gqa/yes_no_data.csv")

In [68]:
data_l = data.iloc[:10000]
data_s = data.iloc[10000:]

In [69]:
data_l["answer"] = data_l["fullAnswer"].to_list()

/tmp/ipykernel_2658725/2876512384.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_l["answer"] = data_l["fullAnswer"].to_list()


In [72]:
data = pd.concat([data_l, data_s])

In [73]:
data

,question,imageId,answer,fullAnswer
0,Are there blankets or curtains?,2346216,"Yes, there is a blanket.","Yes, there is a blanket."
1,Do you see any faucets or coffee pots in the p...,2369120,"Yes, there is a coffee pot.","Yes, there is a coffee pot."
2,Is the woman wearing a skirt?,2408218,"Yes, the woman is wearing a skirt.","Yes, the woman is wearing a skirt."
3,Does he look young?,2403660,"Yes, the man is young.","Yes, the man is young."
4,Are there nightstands beside the bed to the ri...,2387182,"Yes, there is a nightstand beside the bed.","Yes, there is a nightstand beside the bed."
...,...,...,...,...
17628,Is that blanket on top of a cabinet?,2349873,no,"No, the blanket is on top of a bed."
17629,Is there a teddy bear by the toy car that is y...,2365147,yes,"Yes, there is a teddy bear by the toy car."
17630,Is the bottle to the left of the fruit the per...,1593257,no,"No, the bottle is to the right of the banana."
17631,Is the fire hydrant on the left part?,2267,yes,"Yes, the fire hydrant is on the left of the im..."


In [74]:
all_res = []
for inx, row in data.iterrows():
   question = row["question"]
   answer = row["answer"]
   image_id = row["imageId"]
   image_path = "/Data2/Arun-UAV/NLP/vision_halu/visual_genome/VG_100K/" + str(image_id) + ".jpg"
   all_res.append({"image_id": image_id, "image_path": image_path, "question": question, "answer": answer, "question_id": str(uuid4())})

In [75]:
total_df = pd.DataFrame(all_res)

In [76]:
total_df

,image_id,image_path,question,answer,question_id
0,2346216,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Are there blankets or curtains?,"Yes, there is a blanket.",f0766969-5b2c-4cb0-9887-a77d33b51a8f
1,2369120,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Do you see any faucets or coffee pots in the p...,"Yes, there is a coffee pot.",c1c69167-66f9-4463-bcd4-b2391af53dce
2,2408218,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Is the woman wearing a skirt?,"Yes, the woman is wearing a skirt.",517a7d49-b6fa-4613-b467-658e94fac751
3,2403660,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Does he look young?,"Yes, the man is young.",038df875-bd51-4294-a215-8fea7b3fc77c
4,2387182,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Are there nightstands beside the bed to the ri...,"Yes, there is a nightstand beside the bed.",c11f0136-15fc-4ef7-8ab6-5a463b99f642
...,...,...,...,...,...
17628,2349873,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Is that blanket on top of a cabinet?,no,95371406-c310-44eb-aa73-3728dd55be22
17629,2365147,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Is there a teddy bear by the toy car that is y...,yes,a6ece0f6-aa2a-46bc-8f8d-dd026520d701
17630,1593257,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Is the bottle to the left of the fruit the per...,no,2ea2e43e-98ec-42b7-9d1f-189849bcbf4b
17631,2267,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Is the fire hydrant on the left part?,yes,65f702d2-1ec6-427a-bbfd-07b5a624a934


In [83]:
pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/hal_detection_head_train_datasets/gqa/gqa_unlabeled_15k_data.csv")

,image_id,image_path,question,answer,question_id
0,2346216,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Are there blankets or curtains?,"Yes, there is a blanket.",f0766969-5b2c-4cb0-9887-a77d33b51a8f
1,2369120,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Do you see any faucets or coffee pots in the p...,"Yes, there is a coffee pot.",c1c69167-66f9-4463-bcd4-b2391af53dce
2,2408218,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Is the woman wearing a skirt?,"Yes, the woman is wearing a skirt.",517a7d49-b6fa-4613-b467-658e94fac751
3,2403660,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Does he look young?,"Yes, the man is young.",038df875-bd51-4294-a215-8fea7b3fc77c
4,2387182,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Are there nightstands beside the bed to the ri...,"Yes, there is a nightstand beside the bed.",c11f0136-15fc-4ef7-8ab6-5a463b99f642
...,...,...,...,...,...
17628,2349873,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Is that blanket on top of a cabinet?,no,95371406-c310-44eb-aa73-3728dd55be22
17629,2365147,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Is there a teddy bear by the toy car that is y...,yes,a6ece0f6-aa2a-46bc-8f8d-dd026520d701
17630,1593257,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Is the bottle to the left of the fruit the per...,no,2ea2e43e-98ec-42b7-9d1f-189849bcbf4b
17631,2267,/Data2/Arun-UAV/NLP/vision_halu/visual_genome/...,Is the fire hydrant on the left part?,yes,65f702d2-1ec6-427a-bbfd-07b5a624a934
